# 05 – Phân tích nhật ký thực nghiệm

Notebook này tổng hợp **toàn bộ lịch sử** từ `outputs/experiment_log.jsonl`:  
- Đếm và phân loại run thành công / thất bại theo experiment  
- Thống kê mean ± std best val Macro-F1 theo seed  
- Phân tích lỗi: loại lỗi phổ biến nhất, tỷ lệ retry  
- Xuất bảng audit CSV để đưa vào báo cáo  

> **Không tự điền số**: mọi số liệu lấy trực tiếp từ log.

## Cell 1 – Load log

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LOG_PATH = PROJECT_ROOT / 'outputs' / 'experiment_log.jsonl'

if not LOG_PATH.exists():
    raise FileNotFoundError(
        f'Chưa có log: {LOG_PATH}\n'
        'Hãy chạy notebook 02_train_experiments.ipynb trước.'
    )

lines = LOG_PATH.read_text(encoding='utf-8').strip().splitlines()
events = pd.DataFrame(json.loads(line) for line in lines if line.strip())
events['timestamp_utc'] = pd.to_datetime(events['timestamp_utc'], utc=True)

print(f'Tổng số events: {len(events)}')
print('Loại events:\n', events['event'].value_counts().to_string())
events.head(3)

## Cell 2 – Thống kê thành công / thất bại

In [ ]:
completed = events[events['event'] == 'run_completed'].copy()
failed    = events[events['event'].isin(['run_failed', 'run_attempt_failed'])].copy()

print(f'✓ Thành công  : {len(completed)} runs')
print(f'✗ Thất bại    : {len(failed)} events (có thể nhiều attempt / run)')

# Thành công theo experiment
if not completed.empty:
    print('\n--- Thành công theo experiment ---')
    print(completed['experiment'].value_counts().to_string())

# Thất bại theo experiment
if not failed.empty:
    print('\n--- Thất bại theo experiment ---')
    print(failed['experiment'].value_counts().to_string())
    print('\n--- Loại lỗi phổ biến ---')
    if 'error_type' in failed.columns:
        print(failed['error_type'].value_counts().to_string())

## Cell 3 – Bảng mean ± std best val Macro-F1 theo experiment

In [ ]:
if completed.empty:
    print('Chưa có run thành công.')
else:
    agg = (
        completed
        .groupby('experiment')['best_val_macro_f1']
        .agg(count='count', mean='mean', std='std', max='max', min='min')
        .round(4)
        .sort_values('mean', ascending=False)
    )
    # Điền std=0 khi chỉ có 1 run
    agg['std'] = agg['std'].fillna(0)
    agg['mean_pm_std'] = agg.apply(
        lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
    )
    print('=== Best Val Macro-F1 theo experiment (all seeds) ===\n')
    print(agg[['count','mean_pm_std','max','min']].to_string())

    agg.to_csv(PROJECT_ROOT / 'outputs' / 'log_aggregate.csv')

    # Pivot: experiment × seed
    if 'run_id' in completed.columns:
        pivot_rows = []
        for _, row in completed.iterrows():
            rid = str(row.get('run_id', ''))
            # run_id dạng E0_simplecnn_seed42
            parts = rid.rsplit('_seed', 1)
            if len(parts) == 2:
                pivot_rows.append({
                    'experiment': row['experiment'],
                    'seed': int(parts[1]),
                    'best_val_macro_f1': row.get('best_val_macro_f1'),
                })
        if pivot_rows:
            pivot_df = pd.DataFrame(pivot_rows)
            pivot_table = pivot_df.pivot_table(
                index='experiment', columns='seed',
                values='best_val_macro_f1'
            ).round(4)
            print('\n=== Pivot: experiment × seed ===\n')
            print(pivot_table.to_string())
            pivot_table.to_csv(PROJECT_ROOT / 'outputs' / 'log_pivot.csv')

## Cell 4 – Biểu đồ so sánh mean ± std

In [ ]:
if not completed.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Bar chart mean ± std ──
    exp_order = agg.index.tolist()
    colors = plt.cm.tab10(range(len(exp_order)))
    axes[0].bar(exp_order, agg['mean'], yerr=agg['std'], capsize=5,
                color=colors, alpha=0.85)
    for i, (exp, row) in enumerate(agg.iterrows()):
        axes[0].text(i, row['mean'] + row['std'] + 0.005,
                     f"{row['mean']:.3f}", ha='center', fontsize=9)
    axes[0].set_ylabel('Best Val Macro-F1')
    axes[0].set_title('Mean ± Std theo experiment (từ log)')
    axes[0].set_xticklabels(
        [e.replace('_simplecnn','\nSimpleCNN')
          .replace('_resnet_frozen','\nFrozen')
          .replace('_resnet_finetune_aug','\nFT+Aug')
          .replace('_resnet_finetune','\nFT')
         for e in exp_order], fontsize=8
    )
    axes[0].grid(alpha=.2, axis='y')

    # ── Pie chart thành công / thất bại ──
    total_runs = len(completed) + len(failed['run_id'].unique() if 'run_id' in failed.columns else [])
    n_fail_runs = len(failed['run_id'].unique()) if 'run_id' in failed.columns else len(failed)
    n_ok  = len(completed)
    n_fail = max(n_fail_runs, len(failed))
    axes[1].pie([n_ok, n_fail],
                labels=[f'Thành công\n({n_ok})', f'Thất bại\n({n_fail})'],
                colors=['#4caf50', '#f44336'], autopct='%1.0f%%',
                startangle=90)
    axes[1].set_title('Tỷ lệ run thành công / thất bại')

    plt.tight_layout()
    out_path = PROJECT_ROOT / 'outputs' / 'log_experiment_comparison.png'
    plt.savefig(out_path, dpi=160)
    plt.show()
    print(f'Đã lưu: {out_path}')

## Cell 5 – Bảng audit đầy đủ (mỗi dòng = 1 run)

In [ ]:
run_rows = []

for _, row in completed.iterrows():
    run_rows.append({
        'run_id'           : row.get('run_id', ''),
        'experiment'       : row.get('experiment', ''),
        'best_val_macro_f1': row.get('best_val_macro_f1', None),
        'timestamp_utc'    : row['timestamp_utc'],
        'status'           : 'completed',
        'error_type'       : '',
    })

for _, row in failed.iterrows():
    run_rows.append({
        'run_id'           : row.get('run_id', ''),
        'experiment'       : row.get('experiment', ''),
        'best_val_macro_f1': None,
        'timestamp_utc'    : row['timestamp_utc'],
        'status'           : f"failed_attempt_{int(row.get('attempt', 0))}",
        'error_type'       : row.get('error_type', ''),
    })

audit_df = (
    pd.DataFrame(run_rows)
    .sort_values(['run_id', 'timestamp_utc'])
    .reset_index(drop=True)
)
print(f'Tổng số dòng audit: {len(audit_df)}')
display(audit_df)
audit_df.to_csv(PROJECT_ROOT / 'outputs' / 'run_audit.csv', index=False)
print(f'Đã lưu: outputs/run_audit.csv')

## Cell 6 – In tóm tắt cho báo cáo

In [ ]:
print('=' * 60)
print('TÓM TẮT THỰC NGHIỆM – dùng để điền báo cáo')
print('=' * 60)
print(f'  Tổng run thành công : {len(completed)}')
print(f'  Tổng attempt thất bại: {len(failed)}')

if not completed.empty:
    best_row = completed.loc[completed['best_val_macro_f1'].idxmax()]
    print(f'\n  Best run tổng thể:')
    print(f'    run_id           : {best_row.get("run_id", "N/A")}')
    print(f'    experiment       : {best_row.get("experiment", "N/A")}')
    print(f'    best_val_macro_f1: {best_row.get("best_val_macro_f1", "N/A"):.4f}')

    print('\n  Mean Val Macro-F1 theo experiment:')
    for exp, row in agg.iterrows():
        print(f'    {exp:<35} {row["mean_pm_std"]}')

print('\nCác file output đã tạo:')
for f in ['log_aggregate.csv', 'log_pivot.csv', 'run_audit.csv', 'log_experiment_comparison.png']:
    p = PROJECT_ROOT / 'outputs' / f
    status = '✓' if p.exists() else '✗'
    print(f'  {status} outputs/{f}')